# LocalFold

One cell, then one link. What opens is the whole of localfold.org - sequences,
ligands, modified residues, templates, every model, the viewer, the plots and
the downloads - and its Fold button runs on **this runtime's GPU**.

**Runtime → Change runtime type → T4 GPU** before you run it.

---

How it works, because the arrangement is not the obvious one: a page's
JavaScript runs where the page is SHOWN, so a LocalFold in this cell would
normally fold on your laptop with the runtime idle. Instead the runtime serves
the page itself - the link is a port on the runtime, proxied by Colab - and
folds for it - LocalFold's own `web/app.js` in a headless Chrome
there, on the card Colab lent you - and the page in the cell asks it. Same
origin, no tunnel, nothing to paste.

Your browser still draws: the viewer, the plots, the sequence strip. That is
what a browser is for.

In [ ]:
#@title LocalFold { display-mode: "form" }
#@markdown Run this cell, then open the link it prints: the whole of
#@markdown localfold.org - sequences, ligands, modified residues, templates,
#@markdown every model, the viewer, the plots and the downloads - with its
#@markdown **Fold button folding on this runtime's GPU**, not on your laptop.
#@markdown
#@markdown Set **Runtime → Change runtime type → T4 GPU** first. The install
#@markdown takes about two minutes the first time and nothing after that.
branch = "main"  #@param {type:"string"}

import json, os, queue, subprocess, sys, threading, time

PORT, REPO = 8710, '/content/localfold'
SETUP = r"""
set -e
say() { echo "[$(( $(date +%s) - T0 ))s] $*"; }
T0=$(date +%s)

# 🔴 EVERY STEP ASKS WHETHER IT IS NEEDED, because a rerun of this cell used to
# pay the whole minute again. A runtime keeps its filesystem between cells and
# often between notebooks; what it does NOT keep is the shell, so the checks
# have to be about the disk rather than about a variable.
#
# 🔴 AND THE NVIDIA USERSPACE IS USUALLY ALREADY THERE. The expensive install
# was libnvidia-gl-<major>, and the thing that actually matters is whether an
# ICD file exists for Vulkan to load - the driver writes one. Asking first
# turns the common case into a test rather than a download.
# 🔴 AND EVERY PROBE IS AN `if`, NOT AN `&&`. Under `set -e` a bare
# `[ -e X ] && flag=0` IS the script's exit status when the test fails - which
# is the case these exist to detect - so the first cold runtime would have
# stopped here having installed nothing, reporting success.
need_icd=1
if [ -e /usr/share/vulkan/icd.d/nvidia_icd.json ]; then need_icd=0; fi
need_chrome=1
if command -v google-chrome > /dev/null 2>&1; then need_chrome=0; fi
need_loader=1
if ldconfig -p 2>/dev/null | grep -q libvulkan.so.1; then need_loader=0; fi
say "icd=$need_icd chrome=$need_chrome loader=$need_loader"

# 🔴 CHROME IS A ZIP, NOT A .deb - AND `chrome-headless-shell`, WHICH IS THE
# HALF WE USE. Measured on a T4: the .deb is 0.65 s to fetch and **21.4 s in
# dpkg**, while Chrome for Testing is 1.8 s down and 11 s to unzip, with no
# package manager, no maintainer scripts and no triggers. Both it and the full
# build report **nvidia / turing / shader-f16** through a served page: a zip is
# not a lesser browser, it is the same binary without dpkg. 261 MB against 393.
# The setup goes 71 s -> ~41 s, because the one thing that MUST go through apt
# - the driver - then runs underneath the unzip instead of after a dpkg.
if [ "$need_chrome" = 1 ]; then
  ( cd /content \
    && wget -q https://storage.googleapis.com/chrome-for-testing-public/153.0.8010.52/linux64/chrome-headless-shell-linux64.zip \
    && unzip -q -o chrome-headless-shell-linux64.zip -d /opt \
    && ln -sf /opt/chrome-headless-shell-linux64/chrome-headless-shell /usr/local/bin/google-chrome ) &
  chrome_download=$!
fi
if [ -d REPO_PATH ]; then
  ( git -C REPO_PATH fetch -q origin BRANCH && git -C REPO_PATH checkout -q FETCH_HEAD ) &
else
  ( git clone -q --depth 1 --branch BRANCH https://github.com/sokrypton/localfold REPO_PATH ) &
fi
repo_clone=$!

# 🔴 THE DRIVER IS UNPACKED, NOT INSTALLED - AND THAT IS 44 s DOWN TO 15.
# Colab has the whole NVIDIA userspace already, in its own /usr/lib64-nvidia
# and at the kernel module's version - what it does NOT have is a
# libGLX_nvidia carrying the Vulkan ICD entry point, so `vulkaninfo` reports
# "Could not get vkCreateInstance" and WebGPU has no adapter. The package
# supplies that, and `dpkg-deb -x` into / supplies it in **11.5 s** where apt
# takes 44.3 - because apt also unpacks libnvidia-compute (335 MB) and
# libnvidia-gpucomp (70 MB), which nothing here loads.
#
# 🔴 INTO `/`, AND WITH NO `LD_LIBRARY_PATH`. The stack that works is MIXED:
# this libGLX against Colab's own companions at the kernel's version. Forcing
# the extracted tree onto the library path makes it all one version and the
# driver answers ERROR_INCOMPATIBLE_DRIVER; putting the files where the linker
# already looks lets it resolve each name the way a real install does. The
# ICD and layer json come out of the package too, so nothing is hand-written.
if [ "$need_loader" = 1 ] || [ "$need_icd" = 1 ] || [ "$need_chrome" = 1 ]; then
  APT="apt-get -qq -o Dpkg::Use-Pty=0 --no-install-recommends"
  # The Vulkan loader, and the four libraries a zipped Chrome asks for that
  # this image lacks - `ldd` named exactly these. The `t64` spelling is
  # Ubuntu 24.04's and the bare one 22.04's.
  X11="libatk1.0-0t64 libatk-bridge2.0-0t64 libatspi2.0-0t64 libxcomposite1"
  X11_OLD="libatk1.0-0 libatk-bridge2.0-0 libatspi2.0-0 libxcomposite1"
  ( $APT install -y libvulkan1 $X11 > /dev/null 2>&1 \
    || $APT install -y libvulkan1 $X11_OLD > /dev/null 2>&1 \
    || { apt-get -qq update > /dev/null 2>&1
         $APT install -y libvulkan1 $X11 > /dev/null 2>&1 \
           || $APT install -y libvulkan1 $X11_OLD > /dev/null 2>&1; } ) &
  small_libs=$!
fi

if [ "$need_icd" = 1 ]; then
  D=$(nvidia-smi --query-gpu=driver_version --format=csv,noheader 2>/dev/null | cut -d. -f1)
  if [ -n "$D" ]; then
    ( cd /content && apt-get -qq download libnvidia-gl-$D > /dev/null 2>&1 \
      && dpkg-deb -x libnvidia-gl-${D}_*.deb / && ldconfig )
    say "driver (unpacked, not installed)"
  fi
fi
wait $small_libs 2>/dev/null
say "vulkan"

if [ "$need_chrome" = 1 ]; then
  wait $chrome_download
  say "chrome $(google-chrome --version 2>/dev/null || echo 'DID NOT INSTALL')"
fi

# The repository, STARTED FIRST AND WAITED FOR LAST. Measured on a T4 runtime
# the clone is 4.7-5.1 s of a 71 s setup and it shares nothing with apt - one
# is git's network, the other is dpkg's lock - so it runs underneath both.
# `vulkan-tools` is NOT installed: the adapter line the service prints is the
# same question, answered by the thing that will actually fold.
wait $repo_clone 2>/dev/null
say "repository"
"""

with open('/content/_localfold_setup.sh', 'w') as handle:
    handle.write(SETUP.replace('REPO_PATH', REPO).replace('BRANCH', branch))
# 🔴 THE LOG IS KEPT AND NOT PRINTED. A reader presses play and wants a
# button; apt's progress is what they read only when there is no button. It is
# printed in full if the setup fails, which is the one time it says anything.
print('setting up…')
_t0 = time.time()
_setup = subprocess.run(['bash', '/content/_localfold_setup.sh'],
                        capture_output=True, text=True)
if _setup.returncode != 0:
    print(_setup.stdout, _setup.stderr)
    raise SystemExit('the setup failed; the log is above')
_t_setup = time.time() - _t0

# 🔴 THE SERVICE SERVES THE PAGE AND DRIVES THE FOLD, which is what makes one
# cell enough: the link below IS index.html on it, so the page and the thing
# that folds are the same origin - nothing to paste, no CORS and no tunnel.
# 🔴 AND A TAB RATHER THAN A FRAME. An iframe in a Colab output is sandboxed,
# has to be handed `allow="webgpu"` by hand, and is given whatever height a
# slider guessed; the page is a whole application and wants a window. The only
# thing the frame bought was not clicking. See tools/colab_backend.py.
def _drain(stream, sink):
    for line in iter(stream.readline, ''):
        sink.put(line.rstrip())

if not globals().get('_localfold_service'):
    _localfold_service = subprocess.Popen(
        [sys.executable, 'tools/colab_backend.py', '--port', str(PORT)],
        cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    _lines = queue.Queue()
    threading.Thread(target=_drain, args=(_localfold_service.stdout, _lines), daemon=True).start()
    TOKEN, ADAPTER = None, None
    _deadline = time.time() + 300
    while time.time() < _deadline and TOKEN is None:
        try:
            said = _lines.get(timeout=5)
        except queue.Empty:
            continue
        if said.startswith('BACKEND '):
            answer = json.loads(said[len('BACKEND '):])
            TOKEN, ADAPTER = answer['token'], answer['gpu']
        elif 'rror' in said:
            print(said)
    if TOKEN is None:
        raise SystemExit('the fold service did not start; run this cell again')

# 🔴 THE ONE THING WORTH SAYING BESIDES THE BUTTON. 'nvidia' and an
# architecture is the card; 'swiftshader' or 'llvmpipe' is the CPU wearing its
# clothes, and every fold after that is a CPU fold that looks exactly like
# success - so that case is loud and the good case is four words.
_card = '%s %s' % (ADAPTER.get('vendor') or '?', ADAPTER.get('architecture') or '')
_fallback = ('swiftshader' in json.dumps(ADAPTER).lower()
             or 'llvmpipe' in json.dumps(ADAPTER).lower())
if _fallback:
    print('*** %s is the CPU renderer, not the card.' % _card,
          'Runtime > Change runtime type > T4 GPU, then run this cell again. ***')

from google.colab.output import eval_js
from IPython.display import HTML, display

base = eval_js('google.colab.kernel.proxyPort(%d)' % PORT)
# 🔴 AND THE BASE MAY OR MAY NOT END IN A SLASH. Measured in a real session:
# it came back as '…prod.colab.dev' and '%sindex.html' made
# '…prod.colab.devindex.html', which the browser reads as a HOSTNAME - so the
# error is 'server IP address could not be found' rather than a 404, and it
# names a host that has never existed.
base = base if base.endswith('/') else base + '/'
# `?backend=colab` is the page being told which machine folds; `t` is the
# token the service requires of every request, and it rides in the URL because
# a page cannot be handed a header by whoever framed it.
page = '%sindex.html?backend=colab&t=%s' % (base, TOKEN)
display(HTML(
    '<div style="font:14px system-ui;padding:14px 16px;border:1px solid #e5e7eb;'
    'border-radius:10px;display:inline-block">'
    '<a href="%s" target="_blank" rel="noopener" '
    'style="font-size:17px;font-weight:600;text-decoration:none">LocalFold &rarr;</a>'
    '<div style="color:#6b7280;margin-top:6px">opens in a new tab &middot; '
    'folds on %s &middot; ready in %.0f s &middot; keep this notebook running</div>'
    '</div>' % (page, _card, time.time() - _t0)))